# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainkhan006/Flyrank-ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Source: FlyRank *The State of AI-Driven SEO in Numbers* (March 2026).

### Finding 1 (p. 6)

Growing pages are 37.6% longer (3.2K vs 2.3K words) and 20% younger (184 vs 230 days) than declining pages (74,187 vs 45,272). Trend = 30d-vs-prev-30d impressions, Up >10% (p. 5). This is an observed cohort contrast by impression-trend label — directional for review queues, not evidence that length or youth produces growth.

**Question:** Was the growing/declining label built only from GSC impressions, or does it share the same 30-day window that also feeds Health Score in that table?

### Finding 2 (p. 29)

On the 61,790-record active-content subset (p. 1), logistic regression reports 71% holdout accuracy separating growing from declining pages. This is measured model behavior on that sample — decision-support for investigation, not a claim that changing a coefficient produces growth.

**Question:** The 71% holdout is reported without a split unit. Was the cut random pages, grouped by brand, or time-aware, so same-client pages could not leak across train and test?

In [8]:
print("Section 1 is paper-only. No warehouse query in this cell — Run all can continue.")

Section 1 is paper-only. No warehouse query in this cell — Run all can continue.


## 2. My model under an honest split (before/after)

I re-run the Week-5 Logistic Regression ranker twice on the same March 2026 page table. The only change is the split unit. The gap between the two rows is the finding.

### Before — random pages (`randomPageSplit`)

I hold out about 20% of pages with `train_test_split` and `random_state=42`. The same client's pages can sit in both train and test, so Search Console habits can leak across the cut. This row is the leaky comparison. It is not the honest number.

On this rebuild, 55 clients appear on both sides of the random cut.

### After — grouped clients (`groupedClientSplit`)

I shuffle unique `client_hash_id` values (seed 42) and hold out about 20% of clients. Client overlap on this run is 0. That is the honest unit here: one month of warehouse rows, many pages per brand. A random page cut is the wrong unit because it does not respect who owns the pages.

### Same modeling rules on both arms

Both arms use one warehouse query, then split in pandas. The label `isTopVisibility` is the top 20% of `marchImpressions` among pages with `avgPosition >= 1`, cut from **that arm's train only**. Features stay the Week-5 set (`avgPosition`, `measuredDayCount`, `positionSpread`, `wordCount`, `gscHistoryDays`, plus three missing flags). CTR, clicks, impressions, trends, hashes, names, and URLs stay out of X. Precision@20, precision@50, and the base rate are measured on the scored test set only.

### What this holdout measured

Grain is 331,437 March pages. Train-only impression cuts were 1,570.0 (random) and about 1,663.4 (grouped).

| split | n train | n test | n scored test | precision@20 | precision@50 | base rate |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| `randomPageSplit` | 265,149 | 66,288 | 34,883 | 0.95 | 0.98 | 0.197689 |
| `groupedClientSplit` | 297,984 | 33,453 | 8,251 | 0.40 | 0.30 | 0.041 |

On this holdout, grouped precision@20 is 0.40 versus 0.95 on the random cut, and precision@50 is 0.30 versus 0.98. The scored-test base rate also changes (0.197 vs 0.041), so the two rows are not the same mix of pages. That gap is observed on this rebuild. It is decision-support for how I read Week 5. It does not prove a Google ranking mechanism, and it does not make the random-split number the honest one.

In [9]:
%pip -q install duckdb huggingface_hub

from google.colab import userdata
import duckdb
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

randomState = 42

hfToken = userdata.get("HF_TOKEN")
if(not hfToken):
    raise SystemExit(
        "HF_TOKEN secret is missing. Turn it on for this notebook. Do not paste the token in a cell."
    )

con = duckdb.connect()
con.execute(
    "CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)",
    [hfToken],
)

rel = "hf://datasets/FlyRank/internship-warehouse"
factMarch = (
    f"read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')"
)
dimContent = f"read_parquet('{rel}/dim_content.parquet')"
dimClients = f"read_parquet('{rel}/dim_clients.parquet')"

print("loading the March page table...")

con.sql(f"""
CREATE OR REPLACE TABLE pageMarch AS
SELECT
 f.client_hash_id,
 f.content_hash_id,
 AVG(CASE WHEN f.gsc_data_available IS TRUE THEN f.gsc_avg_position END)
 AS avgPosition,
 SUM(CASE WHEN f.gsc_data_available IS TRUE THEN 1 ELSE 0 END)
 AS measuredDayCount,
 STDDEV_SAMP(CASE WHEN f.gsc_data_available IS TRUE THEN f.gsc_avg_position END)
 AS positionSpread,
 MAX(c.word_count) AS wordCount,
 DATE_DIFF('day', MAX(cl.gsc_data_start), DATE '2026-03-01') AS gscHistoryDays,
 SUM(f.gsc_impressions) AS marchImpressions
FROM {factMarch} f
LEFT JOIN {dimContent} c
 ON f.content_hash_id = c.content_hash_id
LEFT JOIN {dimClients} cl
 ON f.client_hash_id = cl.client_hash_id
GROUP BY f.client_hash_id, f.content_hash_id
""")

grain = con.sql("""
SELECT
 COUNT(*) AS pageRows,
 COUNT(DISTINCT content_hash_id) AS distinctPages
FROM pageMarch
""").df()
print("grain check — one page in March 2026")
print(grain.to_string(index=False))

nPages = int(grain.loc[0, "pageRows"])
if(nPages != 331437):
    raise SystemExit(
        "grain is " + str(nPages) + " pages, not 331437 — stopping before the split"
    )

pageMarch = con.sql("SELECT * FROM pageMarch").df()

pageMarch["missingWordCount"] = pageMarch["wordCount"].isna().astype(int)
pageMarch["missingPositionSpread"] = pageMarch["positionSpread"].isna().astype(int)
pageMarch["missingGscHistory"] = pageMarch["gscHistoryDays"].isna().astype(int)
pageMarch["hasRealPosition"] = (
    pageMarch["avgPosition"].notna() & (pageMarch["avgPosition"] >= 1)
)

if("isTopVisibility" in pageMarch.columns):
    raise SystemExit("label was cut on all rows before the split — stopping")

featureCols = [
    "avgPosition",
    "measuredDayCount",
    "positionSpread",
    "wordCount",
    "gscHistoryDays",
    "missingWordCount",
    "missingPositionSpread",
    "missingGscHistory",
]
labelCol = "isTopVisibility"

blockedInX = {
    "marchImpressions",
    "impressions",
    "ctr",
    "clicks",
    "client_hash_id",
    "content_hash_id",
}
blockedHit = False
for col in featureCols:
    if(col in blockedInX):
        blockedHit = True
if(blockedHit):
    raise SystemExit("blocked column landed in X — stopping")

def precisionAtK(labels, scores, k):
    labels = np.asarray(labels)
    scores = np.asarray(scores)
    order = np.argsort(-scores, kind="mergesort")
    return float(labels[order][:k].mean())

def addTrainOnlyLabel(trainPages, testPages):
    trainScoredForCut = trainPages.loc[trainPages["hasRealPosition"]]
    impressionCut = trainScoredForCut["marchImpressions"].quantile(0.80)
    trainLabeled = trainPages.copy()
    testLabeled = testPages.copy()
    trainLabeled[labelCol] = (
        trainLabeled["hasRealPosition"]
        & (trainLabeled["marchImpressions"] >= impressionCut)
    ).astype(int)
    testLabeled[labelCol] = (
        testLabeled["hasRealPosition"]
        & (testLabeled["marchImpressions"] >= impressionCut)
    ).astype(int)
    return trainLabeled, testLabeled, impressionCut

def fitRankerOnScored(trainPages, testPages):
    trainScored = trainPages.loc[trainPages["hasRealPosition"]].copy()
    testScored = testPages.loc[testPages["hasRealPosition"]].copy()
    xTrain = trainScored[featureCols].copy()
    yTrain = trainScored[labelCol].to_numpy()
    xTest = testScored[featureCols].copy()
    yTest = testScored[labelCol].to_numpy()
    imputer = SimpleImputer(strategy="median")
    xTrainFilled = imputer.fit_transform(xTrain)
    xTestFilled = imputer.transform(xTest)
    ranker = LogisticRegression(max_iter=1000, random_state=randomState)
    ranker.fit(xTrainFilled, yTrain)
    testScored["modelScore"] = ranker.predict_proba(xTestFilled)[:, 1]
    p20 = precisionAtK(yTest, testScored["modelScore"], 20)
    p50 = precisionAtK(yTest, testScored["modelScore"], 50)
    baseRate = float(np.mean(yTest))
    return ranker, imputer, testScored, p20, p50, baseRate, xTestFilled, yTest

print("building the leaky random page split...")
trainRandom, testRandom = train_test_split(
    pageMarch, test_size=0.20, random_state=42
)
randomOverlap = len(
    set(trainRandom["client_hash_id"]) & set(testRandom["client_hash_id"])
)
print("random-split client overlap (leaky if above 0):", randomOverlap)
trainRandom, testRandom, randomCut = addTrainOnlyLabel(trainRandom, testRandom)
print("random-split train-only impression cut:", randomCut)
print("fitting the random-split ranker...")
(
    randomRanker,
    randomImputer,
    randomTestScored,
    randomP20,
    randomP50,
    randomBaseRate,
    randomXTestFilled,
    randomYTest,
) = fitRankerOnScored(trainRandom, testRandom)

print("shuffling unique clients with seed 42 and holding out about 20%...")
rng = np.random.RandomState(randomState)
clientIds = pageMarch["client_hash_id"].drop_duplicates().to_numpy()
rng.shuffle(clientIds)
nTestClients = int(round(len(clientIds) * 0.20))
testClientSet = set(clientIds[:nTestClients])
trainClientSet = set(clientIds[nTestClients:])
trainGrouped = pageMarch[pageMarch["client_hash_id"].isin(trainClientSet)].copy()
testGrouped = pageMarch[pageMarch["client_hash_id"].isin(testClientSet)].copy()
clientOverlap = len(trainClientSet & testClientSet)
print("client overlap (must be 0):", clientOverlap)
if(clientOverlap != 0):
    raise SystemExit("client overlap is not 0 — stopping")

trainGrouped, testGrouped, groupedCut = addTrainOnlyLabel(trainGrouped, testGrouped)
print("grouped-split train-only impression cut:", groupedCut)
print("fitting the grouped-client ranker...")
(
    groupedRanker,
    groupedImputer,
    groupedTestScored,
    groupedP20,
    groupedP50,
    groupedBaseRate,
    groupedXTestFilled,
    groupedYTest,
) = fitRankerOnScored(trainGrouped, testGrouped)

comparison = pd.DataFrame(
    [
        {
            "splitName": "randomPageSplit",
            "nTrain": len(trainRandom),
            "nTest": len(testRandom),
            "nScoredTest": len(randomTestScored),
            "precisionAt20": randomP20,
            "precisionAt50": randomP50,
            "baseRate": randomBaseRate,
        },
        {
            "splitName": "groupedClientSplit",
            "nTrain": len(trainGrouped),
            "nTest": len(testGrouped),
            "nScoredTest": len(groupedTestScored),
            "precisionAt20": groupedP20,
            "precisionAt50": groupedP50,
            "baseRate": groupedBaseRate,
        },
    ]
)
print("two-split comparison — scored test set only")
print(comparison.to_string(index=False))
print("grouped test scored frame and ranker are in memory for the leakage audit...")

loading the March page table...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

grain check — one page in March 2026
 pageRows  distinctPages
   331437         331437
building the leaky random page split...
random-split client overlap (leaky if above 0): 55
random-split train-only impression cut: 1570.0
fitting the random-split ranker...
shuffling unique clients with seed 42 and holding out about 20%...
client overlap (must be 0): 0
grouped-split train-only impression cut: 1663.3999999999942
fitting the grouped-client ranker...
two-split comparison — scored test set only
         splitName  nTrain  nTest  nScoredTest  precisionAt20  precisionAt50  baseRate
   randomPageSplit  265149  66288        34883           0.95           0.98  0.197689
groupedClientSplit  297984  33453         8251           0.40           0.30  0.041328
grouped test scored frame and ranker are in memory for the leakage audit...


## 3. Leakage audit

I attack the eight submitted features on the same grouped holdout as Section 2 (`groupedTestScored`, 8,251 scored pages, client overlap 0). I do not train a new architecture.

### Membership check

`featureCols` is `avgPosition`, `measuredDayCount`, `positionSpread`, `wordCount`, `gscHistoryDays`, `missingWordCount`, `missingPositionSpread`, `missingGscHistory`.

`marchImpressions`, clicks, CTR, hash ids, and Health Score / product-flag names are not in X. The submitted set does not include the impression label.

### Overlapping March windows

Every feature and the label come from March 2026. That is disclosed, not hidden. A strictly earlier time window would be a legal time feature; this notebook does not build one.

### Too-good feature: measuredDayCount

Permutation importance on the grouped test (ROC AUC, 5 repeats, seed 42, full 8,251 rows) ranks `measuredDayCount` first (mean drop 0.297), then `positionSpread` (0.121). `measuredDayCount` is same-month GSC presence — a sibling of the impression label, not the label itself.

### With vs without measuredDayCount

Same grouped split, same train-only impression cut, same logistic regression.

| X | precision@20 | precision@50 | base rate |
|---|---:|---:|---:|
| all eight features | 0.40 | 0.30 | 0.041 |
| without `measuredDayCount` | 0.05 | 0.06 | 0.041 |

Dropping `measuredDayCount` collapses precision@20 from 0.40 to 0.05 on this holdout (base rate 0.041). I do not hide that drop. The grouped number with all eight features is still the honest reported number; the drop is measured evidence that same-month GSC day count carries most of the ranking signal here.

### Deliberate leak, then removed

I temporarily add `marchImpressions` to X, fit, and watch precision@20/@50 jump to 1.00 / 1.00 (base rate still 0.041). That is a warning, not a submitted model. I then drop impressions. After the test, `featureCols` is the original eight columns again. The honest grouped number remains 0.40 / 0.30.

### Split unit

The random page split (0.95 / 0.98, 55-client overlap) is the leaky comparison in Section 2. The grouped client split (0.40 / 0.30, overlap 0) is the honest number I audit here. I do not re-run both arms.

### Error examples (hashes only)

Observed on this grouped holdout — not a claim about search ranking.

- False watch (high score, label 0): `client_b10cb2997d0c7c86` / `content_58ab5910b965bcea`
- Missed watch (label 1, low score): `client_1a730cb2640a1abf` / `content_0350eb57034d184f`
- Missed watch (label 1, low score): `client_1a730cb2640a1abf` / `content_c144d5e48cd92f07`

In [10]:
from sklearn.inspection import permutation_importance
import numpy as np
import pandas as pd

neededNames = [
    "featureCols",
    "labelCol",
    "groupedTestScored",
    "groupedRanker",
    "groupedXTestFilled",
    "groupedYTest",
    "groupedP20",
    "groupedP50",
    "groupedBaseRate",
    "pageMarch",
    "precisionAtK",
    "fitRankerOnScored",
    "addTrainOnlyLabel",
]
missingNames = [name for name in neededNames if name not in globals()]
if(len(missingNames) > 0):
    raise SystemExit(
        "kernel is missing " + ", ".join(missingNames) + " — re-run Cell 4 only if the kernel died"
    )

if(("trainGrouped" not in globals()) or ("testGrouped" not in globals())):
    print("grouped train is not named in memory — reconstructing the same client split with seed 42...")
    rng = np.random.RandomState(42)
    clientIds = pageMarch["client_hash_id"].drop_duplicates().to_numpy()
    rng.shuffle(clientIds)
    nTestClients = int(round(len(clientIds) * 0.20))
    testClientSet = set(clientIds[:nTestClients])
    trainClientSet = set(clientIds[nTestClients:])
    trainGrouped = pageMarch[pageMarch["client_hash_id"].isin(trainClientSet)].copy()
    testGrouped = pageMarch[pageMarch["client_hash_id"].isin(testClientSet)].copy()
    trainGrouped, testGrouped, groupedCutReplay = addTrainOnlyLabel(trainGrouped, testGrouped)
    print("reconstructed train-only impression cut:", groupedCutReplay)

print("submitted feature columns:")
print(featureCols)

blockedInX = {
    "marchImpressions",
    "impressions",
    "ctr",
    "clicks",
    "client_hash_id",
    "content_hash_id",
    "healthScore",
    "health_score",
    "Health Score",
}
blockedHit = []
for col in featureCols:
    if(col in blockedInX):
        blockedHit.append(col)
print("blocked columns found in X:", blockedHit if blockedHit else "none")
if(len(blockedHit) > 0):
    raise SystemExit("blocked column landed in X — aborting")

decisionLike = []
for col in featureCols:
    lowered = col.lower()
    if(("health" in lowered) or ("product" in lowered) or ("flag" in lowered)):
        decisionLike.append(col)
print("health-score or product-flag names in X:", decisionLike if decisionLike else "none")

print("scoring feature importance on the honest holdout...")
permResult = permutation_importance(
    groupedRanker,
    groupedXTestFilled,
    groupedYTest,
    n_repeats=5,
    random_state=42,
    scoring="roc_auc",
)
permTable = pd.DataFrame(
    {
        "feature": featureCols,
        "importanceMean": permResult.importances_mean,
        "importanceStd": permResult.importances_std,
    }
).sort_values("importanceMean", ascending=False)
print(permTable.to_string(index=False))

print("honest grouped precision with all features...")
print(
    "precision@20:", groupedP20,
    "precision@50:", groupedP50,
    "base rate:", groupedBaseRate,
)

print("training without measured day count...")
honestFeatureCols = list(featureCols)
featureCols = [col for col in honestFeatureCols if col != "measuredDayCount"]
(
    withoutDaysRanker,
    withoutDaysImputer,
    withoutDaysTestScored,
    withoutP20,
    withoutP50,
    withoutBaseRate,
    withoutXTestFilled,
    withoutYTest,
) = fitRankerOnScored(trainGrouped, testGrouped)
print(
    "without measured day count — precision@20:", withoutP20,
    "precision@50:", withoutP50,
    "base rate:", withoutBaseRate,
)
featureCols = list(honestFeatureCols)
print("feature columns restored after the drop test:", featureCols)

print("fitting a leaky ranker with march impressions in X — this is a warning, not the submitted model...")
featureCols = list(honestFeatureCols) + ["marchImpressions"]
(
    leakyRanker,
    leakyImputer,
    leakyTestScored,
    leakP20,
    leakP50,
    leakBaseRate,
    leakXTestFilled,
    leakYTest,
) = fitRankerOnScored(trainGrouped, testGrouped)
print(
    "WARNING leaky precision@20:", leakP20,
    "precision@50:", leakP50,
    "base rate:", leakBaseRate,
)
featureCols = list(honestFeatureCols)
if("marchImpressions" in featureCols):
    raise SystemExit("marchImpressions is still in the submitted feature set — aborting")
print("impressions removed. submitted feature columns:", featureCols)
print(
    "honest grouped number again — precision@20:", groupedP20,
    "precision@50:", groupedP50,
    "base rate:", groupedBaseRate,
)

hashCols = []
for col in ["client_hash_id", "content_hash_id"]:
    if(col in groupedTestScored.columns):
        hashCols.append(col)
if(len(hashCols) == 0):
    raise SystemExit("no hash-only id columns on groupedTestScored — stop")

print("error examples from grouped scored test — hashes only, no names or URLs...")
falseWatch = groupedTestScored.loc[groupedTestScored[labelCol] == 0].nlargest(1, "modelScore")
missedWatch = groupedTestScored.loc[groupedTestScored[labelCol] == 1].nsmallest(2, "modelScore")
print("false watch — high score, label 0 on this holdout")
print(falseWatch[hashCols + ["modelScore", labelCol]].to_string(index=False))
print("missed watch — label 1, low score on this holdout")
print(missedWatch[hashCols + ["modelScore", labelCol]].to_string(index=False))

submitted feature columns:
['avgPosition', 'measuredDayCount', 'positionSpread', 'wordCount', 'gscHistoryDays', 'missingWordCount', 'missingPositionSpread', 'missingGscHistory']
blocked columns found in X: none
health-score or product-flag names in X: none
scoring feature importance on the honest holdout...
              feature  importanceMean  importanceStd
     measuredDayCount        0.296777       0.007762
       positionSpread        0.120895       0.009811
missingPositionSpread        0.054761       0.006705
          avgPosition        0.009791       0.001473
    missingGscHistory        0.000000       0.000000
     missingWordCount        0.000000       0.000000
            wordCount       -0.002716       0.000477
       gscHistoryDays       -0.005271       0.000224
honest grouped precision with all features...
precision@20: 0.4 precision@50: 0.3 base rate: 0.0413283238395346
training without measured day count...
without measured day count — precision@20: 0.05 precision@50: 0

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 4. Claim rewrite

I rewrite the loudest sentences from Week 5 and from the random split in this notebook. Safe language only: observed, measured, associated, directional, decision-support, precision@K on this holdout.

### What I claimed too strongly

Week 5 treated Logistic Regression versus inverted-position as a ranking win on one grouped holdout — the beats-position energy. This notebook then made random precision@20/@50 of 0.95 / 0.98 look like the honest number. Wording such as "the model finds pages to watch" is what I rewrite here.

### What this notebook measured

The next cell reprints the Section 2 comparison so this rewrite sits next to a number. Random is the leaky comparison (55-client overlap). Grouped is the honest number on this holdout (overlap 0). Base rates differ (0.197 vs 0.041), so the two rows are not the same mix.

| split | precision@20 | precision@50 | base rate | client overlap |
|---|---:|---:|---:|---:|
| randomPageSplit | 0.95 | 0.98 | 0.197 | 55 |
| groupedClientSplit | 0.40 | 0.30 | 0.041 | 0 |

Dropping `measuredDayCount` on the same grouped holdout collapsed precision@20 / @50 to 0.05 / 0.06. A deliberate `marchImpressions` leak hit 1.00 / 1.00; that number is not submitted.

### Rewritten claims

1. On this rebuild, random precision@20/@50 is 0.95 / 0.98 and grouped is 0.40 / 0.30. I treat that gap as observed on this holdout.
2. Grouped precision@20 of 0.40 versus a base rate of 0.041 is measured on this holdout. I use it as decision-support, not a ranking guarantee.
3. Dropping `measuredDayCount` collapsed precision@20 to 0.05 (near the 0.041 base rate). The honest 8-feature number is associated with same-month GSC presence.
4. A deliberate impressions leak hit precision@20/@50 of 1.00 / 1.00. That number is not submitted.

### What I am not claiming

I do not describe search-engine internals. I do not forecast future ranking outcomes. I do not treat this holdout as a universal result. I do not treat any feature as generating impressions. I do not treat the 8-feature ranker as working without `measuredDayCount`.

In [11]:
print("reprinting the Section 2 comparison so the rewrite sits next to a number...")

if("comparison" in globals()):
    print(comparison.to_string(index=False))
else:
    print("comparison is not in memory — reprinting the locked Section 2 table, not a new experiment...")
    comparisonReprint = pd.DataFrame(
        [
            {
                "splitName": "randomPageSplit",
                "precisionAt20": 0.95,
                "precisionAt50": 0.98,
                "baseRate": 0.197,
            },
            {
                "splitName": "groupedClientSplit",
                "precisionAt20": 0.40,
                "precisionAt50": 0.30,
                "baseRate": 0.041,
            },
        ]
    )
    print(comparisonReprint.to_string(index=False))

print("reprinting the honest grouped metrics next to the without-measuredDayCount pair...")

if("groupedP20" in globals() and "groupedP50" in globals() and "groupedBaseRate" in globals()):
    print(
        "honest grouped — precision@20:", groupedP20,
        "precision@50:", groupedP50,
        "base rate:", groupedBaseRate,
    )
else:
    print("honest grouped — precision@20: 0.40 precision@50: 0.30 base rate: 0.041328 — reprint from Section 2, not a new experiment")

if("withoutP20" in globals() and "withoutP50" in globals() and "withoutBaseRate" in globals()):
    print(
        "without measured day count — precision@20:", withoutP20,
        "precision@50:", withoutP50,
        "base rate:", withoutBaseRate,
    )
else:
    print("without measured day count — precision@20: 0.05 precision@50: 0.06 — reprint from Section 3, not a new experiment")

if("leakP20" in globals() and "leakP50" in globals()):
    print(
        "deliberate impressions leak (not submitted) — precision@20:", leakP20,
        "precision@50:", leakP50,
    )

reprinting the Section 2 comparison so the rewrite sits next to a number...
         splitName  nTrain  nTest  nScoredTest  precisionAt20  precisionAt50  baseRate
   randomPageSplit  265149  66288        34883           0.95           0.98  0.197689
groupedClientSplit  297984  33453         8251           0.40           0.30  0.041328
reprinting the honest grouped metrics next to the without-measuredDayCount pair...
honest grouped — precision@20: 0.4 precision@50: 0.3 base rate: 0.0413283238395346
without measured day count — precision@20: 0.05 precision@50: 0.06 base rate: 0.0413283238395346
deliberate impressions leak (not submitted) — precision@20: 1.0 precision@50: 1.0


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

The last box stays empty until this tab is saved to GitHub. This Colab still shows **Save in GitHub to keep changes**. Do not submit the internship card until `work/notebooks/w06_validation_audit.ipynb` exists on the repo.